In [2]:
# Part 3b - PySpark reproduction of the key Polars steps (BFT65B)
# 1) Java 17 (Spark runs on the Java virtual machine); refresh the package list first
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!java -version

# 2) PySpark and Polars (a pip warning about 'dataproc-spark-connect' can be ignored)
!pip -q install pyspark==3.5.3 polars

# 3) Environment of this Colab machine (for the report)
import platform, pyspark, polars
print("PySpark", pyspark.__version__, "| Polars", polars.__version__)
print(platform.platform(), "| CPU cores:", os.cpu_count())
!free -g | head -2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
PySpark 3.5.3 | Polars 1.35.2
Linux-6.6.159+-x86_64-with-glibc2.39 | CPU cores: 2
               total        used        free      shared  buff/cache   available
Mem:              12           1           4           0           7          11


In [3]:
# Point JAVA_HOME to an installed Java (17 preferred, otherwise 21)
import glob, os
cands = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk*")) or sorted(glob.glob("/usr/lib/jvm/java-21-openjdk*"))
os.environ["JAVA_HOME"] = cands[0]
print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [5]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Bigdata_final_project"
!ls -la "$BASE"
# copy to the local disk so both engines read from the same fast disk
!cp "$BASE/spark_input.parquet" /content/spark_input.parquet
!cp "$BASE/part3_spark_reference_by_year.csv" /content/ref_by_year.csv
!ls -la /content/*.parquet /content/*.csv


Mounted at /content/drive
total 60360
drwx------ 2 root root     4096 Nov 23 17:31 .
drwx------ 5 root root     4096 Nov 23 17:40 ..
-rw------- 1 root root     2626 Nov 23 10:58 part3_spark_reference_by_year.csv
-rw------- 1 root root 61797076 Nov 23 10:58 spark_input.parquet
-rw------- 1 root root     2626 Nov 23 17:41 /content/ref_by_year.csv
-rw------- 1 root root 61797076 Nov 23 17:41 /content/spark_input.parquet


In [6]:
import time
from pyspark.sql import SparkSession, functions as F, Window

t0 = time.perf_counter()
spark = (SparkSession.builder.master("local[*]").appName("bft65b-part3")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
SPARK_STARTUP_S = time.perf_counter() - t0
print("Spark", spark.version, "| parallelism:", spark.sparkContext.defaultParallelism,
      f"| startup: {SPARK_STARTUP_S:.1f} s")

Spark 3.5.3 | parallelism: 2 | startup: 6.5 s


In [7]:
def spark_pipeline(path="/content/spark_input.parquet"):
    df = spark.read.parquet(path)
    w = Window.partitionBy("CERT").orderBy("QIDX")          # per-bank, time-ordered window
    df = (df
          .withColumn("F_ROA_Q", F.when(F.col("ASSET") > 0, F.col("NETINC_QTR") * 4 / F.col("ASSET") * 100))
          .withColumn("QIDX_L4", F.lag("QIDX", 4).over(w))
          .withColumn("ASSET_L4", F.lag("ASSET", 4).over(w))
          .withColumn("F_TA_GROWTH_4Q",
                      F.when(((F.col("QIDX") - F.col("QIDX_L4")) == 4) & (F.col("ASSET_L4") > 0),
                             F.col("ASSET") / F.col("ASSET_L4") - 1)))
    med = df.groupBy("REPDTE").agg(F.median("F_ROA_Q").alias("ROA_MEDIAN_Q"))   # 170 rows only
    df = (df.join(F.broadcast(med), on="REPDTE", how="left")
            .withColumn("F_ROA_VS_MEDIAN", F.col("F_ROA_Q") - F.col("ROA_MEDIAN_Q"))
            .withColumn("YEAR", F.year("REPDTE")))
    by_year = (df.groupBy("YEAR")
                 .agg(F.count("*").alias("rows"), F.sum("Y").alias("sum_Y"),
                      F.count("F_TA_GROWTH_4Q").alias("n_growth"),
                      F.avg("F_TA_GROWTH_4Q").alias("mean_growth"),
                      F.avg("F_ROA_VS_MEDIAN").alias("mean_roa_vs_median"))
                 .orderBy("YEAR"))
    return df, by_year

df_sp, by_year_sp = spark_pipeline()
by_year_sp.show(5)

+----+-----+-----+--------+-------------------+--------------------+
|YEAR| rows|sum_Y|n_growth|        mean_growth|  mean_roa_vs_median|
+----+-----+-----+--------+-------------------+--------------------+
|1984|71760|  557|       0|               NULL| -0.3055917811998753|
|1985|72091|  770|   69755| 0.1353400461470484|-0.42614580482210324|
|1986|72182|  943|   69795| 0.1339037199823097| -0.5761926762698879|
|1987|70398| 1421|   69056| 0.1376470011696522|  -0.633743780044935|
|1988|67711| 2330|   66404|0.12817492161235486| -0.6717180899983562|
+----+-----+-----+--------+-------------------+--------------------+
only showing top 5 rows



In [9]:
import numpy as np, pandas as pd

res = by_year_sp.toPandas()
ref = pd.read_csv("/content/ref_by_year.csv")
m = ref.merge(res, on="YEAR", suffixes=("_polars", "_spark"))
rows = []
for c in ["rows", "sum_Y", "n_growth", "mean_growth", "mean_roa_vs_median"]:
    a = m[f"{c}_polars"].astype(float).fillna(0)
    b = m[f"{c}_spark"].astype(float).fillna(0)
    rows.append({"check": f"by year: {c}", "max_abs_diff": float(np.max(np.abs(a - b))),
                 "match": bool(np.allclose(a, b, rtol=1e-9, atol=1e-9))})

tot = df_sp.agg(F.count("*").alias("rows"), F.sum("Y").alias("sum_Y"),
                F.count("F_TA_GROWTH_4Q").alias("n_growth"),
                F.sum("F_TA_GROWTH_4Q").alias("sum_growth"),
                F.sum("F_ROA_VS_MEDIAN").alias("sum_roa_vs_median")).toPandas().iloc[0]
polars_ref = {"rows": 1682738, "sum_Y": 12495, "n_growth": 1587806,
              "sum_growth": 223494.07276020735, "sum_roa_vs_median": 2770916.428857274}
for k, v in polars_ref.items():
    rows.append({"check": f"total: {k}", "polars": v, "spark": float(tot[k]),
                 "max_abs_diff": abs(float(tot[k]) - v),
                 "match": bool(np.isclose(float(tot[k]), v, rtol=1e-9, atol=1e-9))})

check = pd.DataFrame(rows)
check.to_csv("/content/part3_spark_check.csv", index=False)
print("ALL MATCH:", bool(check["match"].all()))
check

ALL MATCH: True


,check,max_abs_diff,match,polars,spark
0,by year: rows,0.000000e+00,True,NaN,NaN
1,by year: sum_Y,0.000000e+00,True,NaN,NaN
2,by year: n_growth,0.000000e+00,True,NaN,NaN
3,by year: mean_growth,6.661338e-16,True,NaN,NaN
4,by year: mean_roa_vs_median,2.842171e-14,True,NaN,NaN
5,total: rows,0.000000e+00,True,1.682738e+06,1.682738e+06
6,total: sum_Y,0.000000e+00,True,1.249500e+04,1.249500e+04
7,total: n_growth,0.000000e+00,True,1.587806e+06,1.587806e+06
8,total: sum_growth,3.259629e-09,True,2.234941e+05,2.234941e+05
9,total: sum_roa_vs_median,1.443550e-08,True,2.770916e+06,2.770916e+06


In [10]:
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print("===== FORMATTED PHYSICAL PLAN (by-year result) =====")
    by_year_sp.explain("formatted")
    print("\n===== EXTENDED PLAN (parsed -> analyzed -> optimized -> physical) =====")
    by_year_sp.explain("extended")
plan_text = buf.getvalue()
open("/content/part3_spark_explain.txt", "w").write(plan_text)
print(plan_text)

===== FORMATTED PHYSICAL PLAN (by-year result) =====
== Physical Plan ==
AdaptiveSparkPlan (50)
+- == Final Plan ==
   * Sort (31)
   +- AQEShuffleRead (30)
      +- ShuffleQueryStage (29), Statistics(sizeInBytes=2.4 KiB, rowCount=43)
         +- Exchange (28)
            +- * HashAggregate (27)
               +- AQEShuffleRead (26)
                  +- ShuffleQueryStage (25), Statistics(sizeInBytes=9.1 KiB, rowCount=129)
                     +- Exchange (24)
                        +- * HashAggregate (23)
                           +- * Project (22)
                              +- * BroadcastHashJoin LeftOuter BuildRight (21)
                                 :- * Project (9)
                                 :  +- Window (8)
                                 :     +- * Sort (7)
                                 :        +- AQEShuffleRead (6)
                                 :           +- ShuffleQueryStage (5), Statistics(sizeInBytes=89.9 MiB, rowCount=1.68E+6)
                         

In [11]:
import statistics, polars as pl

def polars_pipeline(path="/content/spark_input.parquet"):
    c = pl.col
    lf = pl.scan_parquet(path).sort(["CERT", "QIDX"]).with_columns(
        pl.when(c("ASSET") > 0).then(c("NETINC_QTR") * 4 / c("ASSET") * 100).alias("F_ROA_Q"),
        pl.when(((c("QIDX") - c("QIDX").shift(4).over("CERT")) == 4) & (c("ASSET").shift(4).over("CERT") > 0))
          .then(c("ASSET") / c("ASSET").shift(4).over("CERT") - 1).alias("F_TA_GROWTH_4Q"))
    med = lf.group_by("REPDTE").agg(c("F_ROA_Q").median().alias("ROA_MEDIAN_Q"))
    return (lf.join(med, on="REPDTE", how="left")
              .with_columns((c("F_ROA_Q") - c("ROA_MEDIAN_Q")).alias("F_ROA_VS_MEDIAN"),
                            c("REPDTE").dt.year().alias("YEAR"))
              .group_by("YEAR").agg(pl.len().alias("rows"), c("Y").sum().alias("sum_Y"),
                                    c("F_TA_GROWTH_4Q").count().alias("n_growth"),
                                    c("F_TA_GROWTH_4Q").mean().alias("mean_growth"),
                                    c("F_ROA_VS_MEDIAN").mean().alias("mean_roa_vs_median"))
              .sort("YEAR").collect())

def bench(fn, repeats=5):
    fn()                                   # warm-up
    times = []
    for _ in range(repeats):
        t = time.perf_counter(); fn(); times.append(time.perf_counter() - t)
    return statistics.median(times), min(times), max(times)

timing = []
for name, fn in [("Spark local[*]", lambda: spark_pipeline()[1].toPandas()),
                 ("Polars lazy", polars_pipeline)]:
    med_s, mn, mx = bench(fn)
    timing.append({"engine": name, "median_s": med_s, "min_s": mn, "max_s": mx, "runs": 5})
timing.append({"engine": "Spark session start-up (once)", "median_s": SPARK_STARTUP_S})
timing = pd.DataFrame(timing)
timing.to_csv("/content/part3_spark_timing.csv", index=False)
timing

,engine,median_s,min_s,max_s,runs
0,Spark local[*],9.821040,7.567555,10.195937,5.0
1,Polars lazy,0.834710,0.818097,0.855745,5.0
2,Spark session start-up (once),6.461597,NaN,NaN,NaN


In [12]:
!cp /content/part3_spark_check.csv /content/part3_spark_explain.txt /content/part3_spark_timing.csv "$BASE/"
!ls -la "$BASE"
print("Saved. Now: File -> Save, then File -> Download -> Download .ipynb")

total 60401
drwx------  2 root root     4096 Nov 23 17:55 .
drwx------ 11 root root     4096 Nov 23 17:40 ..
-rw-------  1 root root      521 Nov 23 17:55 part3_spark_check.csv
-rw-------  1 root root    40074 Nov 23 17:55 part3_spark_explain.txt
-rw-------  1 root root     2626 Nov 23 10:58 part3_spark_reference_by_year.csv
-rw-------  1 root root      227 Nov 23 17:55 part3_spark_timing.csv
-rw-------  1 root root 61797076 Nov 23 10:58 spark_input.parquet
Saved. Now: File -> Save, then File -> Download -> Download .ipynb
